# 4장 02. 운영 숫자 비교

이 notebook은 baseline 로그와 current 로그를 같은 방식으로 요약합니다.


## 기본 개념: operational metric과 baseline 비교

Operational metric은 많은 요청을 집계해 현재 시스템 상태를 보여 주는 숫자입니다. 모델 metric이 label과 prediction을 비교한다면, 운영 metric은 error rate, latency, request count, prediction distribution처럼 서비스가 실제로 어떻게 동작하는지 보여 줍니다.

Baseline은 비교 기준입니다. current error rate가 5%라는 숫자만 보면 높은지 낮은지 알기 어렵습니다. 평소 0.1%였는지 4.8%였는지에 따라 해석이 달라집니다.

MLOps monitoring에서는 모델 출력과 시스템 상태를 함께 봐야 합니다. score 평균과 `high_risk` 비율이 움직였는데 latency와 validation failure도 함께 증가했다면, 모델 자체 변화만이 아니라 입력 계약, API 오류, 운영 부하를 같이 확인해야 합니다.

| Metric | 설명 | 해석 주의점 |
| --- | --- | --- |
| error rate | 실패 요청 비율 | 모델 품질과 API 오류를 구분해야 함 |
| latency | 응답 시간 | 모델 계산, network, dependency 지연이 섞일 수 있음 |
| average score | 모델 출력 평균 | label 없이 품질 저하를 확정할 수 없음 |
| prediction rate | class 비율 | threshold와 입력 분포 영향을 함께 받음 |
| validation failure | 입력 계약 실패 | client/source 문제 후보 |

이 노트북은 Grafana를 대체하지 않습니다. 대시보드에 올라갈 숫자가 어떻게 계산되는지 작은 sample로 먼저 이해하는 단계입니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
import json
from pathlib import Path

base_path = Path("artifacts/logs/chapter_04_normal_events.jsonl")
current_path = Path("artifacts/logs/chapter_04_anomaly_events.jsonl")
if not base_path.exists():
    base_path = Path("../artifacts/logs/chapter_04_normal_events.jsonl")
    current_path = Path("../artifacts/logs/chapter_04_anomaly_events.jsonl")
if not base_path.exists():
    base_path = Path("../../artifacts/logs/chapter_04_normal_events.jsonl")
    current_path = Path("../../artifacts/logs/chapter_04_anomaly_events.jsonl")


## 1. 두 로그 파일 읽기

각 줄은 요청 이벤트 하나입니다.


In [ ]:
baseline = [json.loads(line) for line in base_path.read_text().splitlines()[:120]]
current = [json.loads(line) for line in current_path.read_text().splitlines()[:120]]

print("baseline events:", len(baseline))
print("current events:", len(current))


## 2. 오류율, high_risk 비율, 평균 latency 비교

복잡한 dashboard 대신 세 숫자만 먼저 봅니다.


In [ ]:
base = pd.DataFrame(baseline)
cur = pd.DataFrame(current)

summary = pd.DataFrame([
    {"dataset": "baseline", "error_rate": base["validation_failure"].mean(), "high_risk_rate": (base["prediction"] == "high_risk").mean(), "avg_latency_ms": base["latency_ms"].mean()},
    {"dataset": "current", "error_rate": cur["validation_failure"].mean(), "high_risk_rate": (cur["prediction"] == "high_risk").mean(), "avg_latency_ms": cur["latency_ms"].mean()},
])
summary.round(4)
